# Pan-DA marker expression across ASAP and Ma spatial samples
* All ASAP Visium samples show very low (if any) expression of the four Pan-DA identity markers -- `TH`, `SLC6A3`, `SLC18A2`, `DDC` (Kamath et al. 2022). In contrast, Ma (2025) samples show intermediate level of expression for the same genes.
* This notebook shows expression for these genes along tissue only and highly expressed genes for comparison. Each sample is coloured with raw UMI (row 1) and log-normalized (row 2) values.
* If local files exist in `_RAW_LOCAL_CACHE_DIR`, they will be loaded. Otherwise they will be downloaded from `_RAW_DATASET_GCS_DIR` (`gcloud` and bucket read credentials are needed).

## Imports

In [ ]:
import sys
import warnings
from pathlib import Path

import anndata as ad
import scanpy as sc

sc.set_figure_params(dpi=100, frameon=False)
warnings.filterwarnings("ignore", message=".*deprecated.*")

## Inputs

In [ ]:
repo_name = "spatial-sc-rnaseq-integration-wf"
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from notebook_utils import get_repo_root  # must come after the utils sys.path.insert above
repo_root = get_repo_root(repo_name)
from genomics_constants import KAMATH_PAN_DA_MARKER_SYMBOLS
from input_definitions import (
    ASAP_RAW_H5AD_SUFFIX,
    MA_SPATIAL_H5AD_SUFFIX,
    asap_raw_gcs_dir,
    asap_raw_local_cache_dir,
    ma_spatial_h5ad_dir,
)
from notebook_utils import ensure_cached, list_available_and_cached, spatial_plot_genes

PAN_DA_MARKERS = list(KAMATH_PAN_DA_MARKER_SYMBOLS)
KNOWN_HIGLY_EXPRESSED = list(["MALAT1"]) # Genes added as plot ceiling control

# ----- ASAP samples ----------
_RAW_DATASET_GCS_DIR = asap_raw_gcs_dir()
_RAW_LOCAL_CACHE_DIR = asap_raw_local_cache_dir(repo_root)
RAW_H5AD_SUFFIX = ASAP_RAW_H5AD_SUFFIX

available_asap_samples, cached_asap_samples, missing_asap_samples = list_available_and_cached(
    _RAW_DATASET_GCS_DIR, _RAW_LOCAL_CACHE_DIR, RAW_H5AD_SUFFIX
)
print(f"=== ASAP samples ===")
print(f"Found {len(available_asap_samples)} sample(s) in {_RAW_DATASET_GCS_DIR}")
print(f"{len(cached_asap_samples)}/{len(available_asap_samples)} already cached locally in {_RAW_LOCAL_CACHE_DIR}")
if missing_asap_samples:
    print(f"{len(missing_asap_samples)} sample(s) will be downloaded automatically below, as each is reached.")

# ----- Ma samples ----------
_MA_SPATIAL_H5AD_DIR = ma_spatial_h5ad_dir(repo_root)
MA_H5AD_SUFFIX = MA_SPATIAL_H5AD_SUFFIX
ma_samples = sorted(p.name.removesuffix(MA_H5AD_SUFFIX) for p in _MA_SPATIAL_H5AD_DIR.glob(f"*{MA_H5AD_SUFFIX}"))
print(f"=== Ma (2025) samples ===")
print(f"Found {len(ma_samples)} sample(s) in {_MA_SPATIAL_H5AD_DIR}")

## Plot Pan-DA markers in ASAP samples

In [ ]:
n_asap_samples_to_plot = 95 # 95 to plot all ASAP samples
for sample_name in available_asap_samples[:n_asap_samples_to_plot]:
    print(f"=== {sample_name} ===")
    try:
        raw_path = ensure_cached(_RAW_DATASET_GCS_DIR, _RAW_LOCAL_CACHE_DIR, sample_name + RAW_H5AD_SUFFIX)

        adata = ad.read_h5ad(raw_path)
        spatial_plot_genes(adata, PAN_DA_MARKERS + KNOWN_HIGLY_EXPRESSED, sample_name)

        # del adata
    except Exception as e:
        print(f"ERROR: {sample_name} failed ({type(e).__name__}: {e}) -- skipping, continuing with the rest.")

## Plot Pan-DA markers in Ma (2025) samples

In [ ]:
n_ma_samples_to_plot = 10 # 10 to plot all Ma samples
for sample_name in ma_samples[:n_ma_samples_to_plot]:
    print(f"=== {sample_name} ===")
    try:
        raw_path = _MA_SPATIAL_H5AD_DIR / (sample_name + MA_H5AD_SUFFIX)

        adata = ad.read_h5ad(raw_path)
        spatial_plot_genes(adata, PAN_DA_MARKERS + KNOWN_HIGLY_EXPRESSED, sample_name)

        del adata
    except Exception as e:
        print(f"ERROR: {sample_name} failed ({type(e).__name__}: {e}) -- skipping, continuing with the rest.")